# CMPE 460 TextCNN Training on SecureAI-SE

**Member 2**

This notebook loads and encodes the pinned SecureAI-SE dataset, then trains a character-level TextCNN. Run all cells from top to bottom. The input text, label order, character vocabulary, and 256-character encoding follow `02_web_data_preprocessing.ipynb`.

Training uses the training split. Validation macro-F1 selects the best epoch and controls early stopping. The architecture uses 32-dimensional embeddings, 64 filters for each kernel size of 3, 4, and 5, dropout of 0.3, and Adam with a learning rate of 0.001.


## 1. Import Packages and Load the Dataset

The cell prints each setup stage. It uses the pinned Parquet files from the Hugging Face cache when available and downloads missing files. Package installation is performed once in your Python environment. If an import fails, run the following command in a separate code cell, restart the kernel, and run this notebook again.

```python
%pip install torch datasets scikit-learn pandas numpy pyarrow huggingface-hub
```

A CUDA-enabled PyTorch installation uses an available NVIDIA GPU. A CPU installation runs the model on the CPU.


In [1]:
from time import perf_counter

setup_started = perf_counter()
print("Step 1/3: importing packages...", flush=True)

import random

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from datasets import Features
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import LocalEntryNotFoundError
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print(f"Imports completed in {perf_counter() - setup_started:.1f}s.", flush=True)
print("Step 2/3: loading dataset files...", flush=True)

DATASET_ID = "SecureAI-SE/http-attack-requests"
DATASET_REVISION = "9ada662402e0620cc4e3bd32b2cc3c2025bb3804"
SPLITS = ("train", "validation", "test")
MAX_LENGTH = 256
PAD_ID = 0
UNK_ID = 1

frames = {}
label_names = []

for split in SPLITS:
    download_options = {
        "repo_id": DATASET_ID,
        "repo_type": "dataset",
        "revision": DATASET_REVISION,
        "filename": f"data/{split}-00000-of-00001.parquet",
    }
    try:
        parquet_path = hf_hub_download(**download_options, local_files_only=True)
        print(f"{split}: reading cached file.", flush=True)
    except LocalEntryNotFoundError:
        print(f"{split}: downloading dataset file...", flush=True)
        parquet_path = hf_hub_download(**download_options)

    table = pq.read_table(parquet_path)
    split_label_names = Features.from_arrow_schema(table.schema)["label"].names
    if split == "train":
        label_names = split_label_names
    assert split_label_names == label_names
    frames[split] = table.to_pandas()[["request", "label"]].copy()

print("Step 3/3: encoding requests...", flush=True)

train_characters = sorted(set("".join(frames["train"]["request"])))
char_to_id = {
    char: index
    for index, char in enumerate(train_characters, start=2)
}


def encode_requests(requests):
    encoded = np.full((len(requests), MAX_LENGTH), PAD_ID, dtype=np.int64)
    for row_index, text in enumerate(requests):
        character_ids = [
            char_to_id.get(char, UNK_ID)
            for char in text[:MAX_LENGTH]
        ]
        encoded[row_index, :len(character_ids)] = character_ids
    return encoded


X = {
    split: encode_requests(frames[split]["request"].tolist())
    for split in SPLITS
}
y = {
    split: frames[split]["label"].to_numpy(dtype=np.int64)
    for split in SPLITS
}

for split in SPLITS:
    assert X[split].shape == (len(y[split]), MAX_LENGTH)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < len(char_to_id) + 2
    print(f"{split}: X={X[split].shape}, y={y[split].shape}", flush=True)

print(f"Data setup completed in {perf_counter() - setup_started:.1f}s.", flush=True)


Step 1/3: importing packages...


c:\Users\yalin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports completed in 6.7s.
Step 2/3: loading dataset files...
train: reading cached file.
validation: reading cached file.
test: reading cached file.
Step 3/3: encoding requests...
train: X=(4900, 256), y=(4900,)
validation: X=(700, 256), y=(700,)
test: X=(1400, 256), y=(1400,)
Data setup completed in 6.9s.


## 2. Configure Training and Create Data Loaders

Each batch contains up to 64 requests. Only the training loader shuffles the rows. The seed, PyTorch version, device, and CPU thread count are printed so that the experiment can be reproduced. The training loop reports progress every 20 batches.


In [2]:
SEED = 42
BATCH_SIZE = 64
EMBEDDING_DIM = 32
FILTERS_PER_KERNEL = 64
KERNEL_SIZES = (3, 4, 5)
DROPOUT = 0.3
LEARNING_RATE = 0.001
MAX_EPOCHS = 15
PATIENCE = 4
LOG_EVERY_BATCHES = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocabulary_size = len(char_to_id) + 2
number_of_classes = len(label_names)

loaders = {}
for split in ("train", "validation"):
    samples = TensorDataset(
        torch.from_numpy(X[split]),
        torch.from_numpy(y[split]),
    )
    loaders[split] = DataLoader(
        samples,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

print(f"Device: {device}", flush=True)
print(f"PyTorch version: {torch.__version__}", flush=True)
print(f"CPU threads: {torch.get_num_threads()}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
else:
    print("This kernel will train on the CPU.", flush=True)
print(f"Dataset revision: {DATASET_REVISION}", flush=True)
print(f"Vocabulary size: {vocabulary_size}", flush=True)
print(f"Number of classes: {number_of_classes}", flush=True)
print(f"Maximum input length: {MAX_LENGTH}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Learning rate: {LEARNING_RATE}", flush=True)
print(f"Maximum epochs: {MAX_EPOCHS}", flush=True)
print(f"Early-stopping patience: {PATIENCE}", flush=True)


Device: cpu
PyTorch version: 2.15.0+cpu
CPU threads: 12
This kernel will train on the CPU.
Dataset revision: 9ada662402e0620cc4e3bd32b2cc3c2025bb3804
Vocabulary size: 73
Number of classes: 7
Maximum input length: 256
Batch size: 64
Learning rate: 0.001
Maximum epochs: 15
Early-stopping patience: 4


## 3. Define the TextCNN

The embedding layer maps each character ID to a learned vector. Three parallel convolutions examine windows of 3, 4, and 5 characters. ReLU and max pooling summarize their outputs. A dropout layer and a linear layer produce seven class scores.

Pooling uses convolution windows containing real characters. Windows that include padding contribute zero after ReLU.

[PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html) · [PyTorch Conv1d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)


In [3]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        number_of_classes,
        embedding_dim=32,
        filters_per_kernel=64,
        kernel_sizes=(3, 4, 5),
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_id,
        )
        self.convolutions = nn.ModuleList([
            nn.Conv1d(embedding_dim, filters_per_kernel, kernel_size)
            for kernel_size in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(
            filters_per_kernel * len(kernel_sizes),
            number_of_classes,
        )

    def forward(self, tokens):
        embedded = self.embedding(tokens).transpose(1, 2)
        nonpadding = tokens.ne(self.pad_id)
        pooled_features = []

        for convolution in self.convolutions:
            features = torch.relu(convolution(embedded))
            kernel_size = convolution.kernel_size[0]
            valid_windows = nonpadding.unfold(1, kernel_size, 1).all(dim=-1)
            features = features.masked_fill(~valid_windows.unsqueeze(1), 0.0)
            pooled_features.append(features.amax(dim=2))

        combined = torch.cat(pooled_features, dim=1)
        return self.classifier(self.dropout(combined))


model = TextCNN(
    vocabulary_size=vocabulary_size,
    number_of_classes=number_of_classes,
    embedding_dim=EMBEDDING_DIM,
    filters_per_kernel=FILTERS_PER_KERNEL,
    kernel_sizes=KERNEL_SIZES,
    dropout=DROPOUT,
    pad_id=PAD_ID,
).to(device)

model.eval()
with torch.no_grad():
    example_inputs = torch.from_numpy(X["train"][:BATCH_SIZE]).to(device)
    example_scores = model(example_inputs)

assert example_scores.shape == (len(example_inputs), number_of_classes)
assert torch.isfinite(example_scores).all()

print(model)
print(f"Example output shape: {tuple(example_scores.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


TextCNN(
  (embedding): Embedding(73, 32, padding_idx=0)
  (convolutions): ModuleList(
    (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(32, 64, kernel_size=(4,), stride=(1,))
    (2): Conv1d(32, 64, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=192, out_features=7, bias=True)
)
Example output shape: (64, 7)
Trainable parameters: 28,455


## 4. Train and Select the Best Epoch

Cross-entropy compares the class scores with the integer labels. Training progress is printed after the first batch, every 20 batches, and the final batch of each epoch. The first batch may take longer while PyTorch initializes its execution kernels.

After each epoch, validation accuracy and macro-F1 are calculated. The best weights are kept in memory. Training stops after four consecutive epochs without an improvement in validation macro-F1. Epoch durations are recorded alongside the metrics.


In [4]:
from copy import deepcopy

from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        scores = model(inputs)
        loss = criterion(scores, targets)

        total_loss += loss.item() * targets.size(0)
        true_labels.extend(targets.cpu().tolist())
        predicted_labels.extend(scores.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            labels=list(range(number_of_classes)),
            average="macro",
            zero_division=0,
        ),
    }
    return metrics, true_labels, predicted_labels


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history_rows = []

print(f"Starting training for up to {MAX_EPOCHS} epochs on {device}.", flush=True)

training_started = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = perf_counter()
    model.train()
    total_training_loss = 0.0

    for batch_index, (inputs, targets) in enumerate(loaders["train"], start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(inputs)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        total_training_loss += loss.item() * targets.size(0)

        if (
            batch_index == 1
            or batch_index % LOG_EVERY_BATCHES == 0
            or batch_index == len(loaders["train"])
        ):
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
                f"batch {batch_index:02d}/{len(loaders['train'])} | "
                f"elapsed {perf_counter() - epoch_started:.1f}s",
                flush=True,
            )

    training_loss = total_training_loss / len(loaders["train"].dataset)
    validation, _, _ = evaluate_model(model, loaders["validation"])
    epoch_seconds = perf_counter() - epoch_started

    history_rows.append({
        "epoch": epoch,
        "train_loss": training_loss,
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["accuracy"],
        "validation_macro_f1": validation["macro_f1"],
        "epoch_seconds": epoch_seconds,
    })

    print(
        f"Epoch {epoch:02d} | train loss {training_loss:.4f} | "
        f"val loss {validation['loss']:.4f} | "
        f"val accuracy {validation['accuracy']:.4f} | "
        f"val macro-F1 {validation['macro_f1']:.4f} | "
        f"time {epoch_seconds:.1f}s",
        flush=True,
    )

    if validation["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation["macro_f1"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}.")
        break

model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(history_rows)

print(f"Restored the best epoch: {best_epoch}")
print(f"Total training time: {perf_counter() - training_started:.1f}s")
display(history.round(4))


Starting training for up to 15 epochs on cpu.
Epoch 01/15 | batch 01/77 | elapsed 0.0s
Epoch 01/15 | batch 20/77 | elapsed 0.6s
Epoch 01/15 | batch 40/77 | elapsed 1.1s
Epoch 01/15 | batch 60/77 | elapsed 1.6s
Epoch 01/15 | batch 77/77 | elapsed 2.0s
Epoch 01 | train loss 1.3127 | val loss 0.5365 | val accuracy 0.8871 | val macro-F1 0.8870 | time 2.1s
Epoch 02/15 | batch 01/77 | elapsed 0.0s
Epoch 02/15 | batch 20/77 | elapsed 0.6s
Epoch 02/15 | batch 40/77 | elapsed 1.0s
Epoch 02/15 | batch 60/77 | elapsed 1.6s
Epoch 02/15 | batch 77/77 | elapsed 2.0s
Epoch 02 | train loss 0.5039 | val loss 0.2433 | val accuracy 0.9429 | val macro-F1 0.9432 | time 2.1s
Epoch 03/15 | batch 01/77 | elapsed 0.0s
Epoch 03/15 | batch 20/77 | elapsed 0.5s
Epoch 03/15 | batch 40/77 | elapsed 1.0s
Epoch 03/15 | batch 60/77 | elapsed 1.5s
Epoch 03/15 | batch 77/77 | elapsed 1.9s
Epoch 03 | train loss 0.2924 | val loss 0.1628 | val accuracy 0.9557 | val macro-F1 0.9557 | time 2.0s
Epoch 04/15 | batch 01/77 | el

,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1,epoch_seconds
0,1,1.3127,0.5365,0.8871,0.8870,2.1274
1,2,0.5039,0.2433,0.9429,0.9432,2.0953
2,3,0.2924,0.1628,0.9557,0.9557,2.0142
3,4,0.2000,0.1247,0.9643,0.9644,1.9460
4,5,0.1548,0.1035,0.9671,0.9671,1.9723
5,6,0.1390,0.0898,0.9729,0.9729,1.9604
6,7,0.1135,0.0785,0.9743,0.9744,1.9977
7,8,0.0994,0.0842,0.9714,0.9714,2.2307
8,9,0.0861,0.0771,0.9657,0.9658,2.0600
9,10,0.0733,0.0773,0.9657,0.9658,2.0856


## 5. Report Validation Results

The following metrics describe the model selected using validation macro-F1. Record them as development results in the architecture comparison. Test metrics will be reported after the experiment settings have been selected.

Save this notebook with its outputs after the run completes. Run all cells again to start a new experiment from the recorded seed.


In [5]:
from sklearn.metrics import classification_report

validation_metrics, validation_labels, validation_predictions = evaluate_model(
    model,
    loaders["validation"],
)

print(f"Best epoch: {best_epoch}")
print(f"Validation accuracy: {validation_metrics['accuracy']:.4f}")
print(f"Validation macro-F1: {validation_metrics['macro_f1']:.4f}")
print()
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=list(range(number_of_classes)),
    target_names=label_names,
    digits=4,
    zero_division=0,
))


Best epoch: 12
Validation accuracy: 0.9771
Validation macro-F1: 0.9772

              precision    recall  f1-score   support

      normal     0.9825    0.9912    0.9868       113
        sqli     1.0000    0.9674    0.9834        92
         xss     0.9804    1.0000    0.9901       100
        ssrf     1.0000    1.0000    1.0000        92
        ssti     1.0000    1.0000    1.0000       102
         lfi     0.9468    0.9271    0.9368        96
   traversal     0.9346    0.9524    0.9434       105

    accuracy                         0.9771       700
   macro avg     0.9777    0.9769    0.9772       700
weighted avg     0.9773    0.9771    0.9771       700

